# Teilauftrag 3: Model
**Datensatz:** Stroke Prediction Dataset (skalierte Version)  
**Ziel:** Klassifikationsmodell trainieren und Vorhersagen treffen

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('stroke_data_scaled.csv')
print('Shape:', df.shape)
df.head()

Shape: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,0.816895,0,1,Yes,Private,Urban,0.801265,0.301260,formerly smoked,1
1,51676,Female,0.743652,0,0,Yes,Self-employed,Rural,0.679023,0.203895,never smoked,1
2,31112,Male,0.975586,0,1,Yes,Private,Rural,0.234512,0.254296,never smoked,1
3,60182,Female,0.597168,0,0,Yes,Private,Urban,0.536008,0.276060,smokes,1
4,1665,Female,0.963379,1,0,Yes,Self-employed,Rural,0.549349,0.156930,never smoked,1


## Aufgabe 3.1 – Train/Test-Aufteilung

Zuerst werden kategorische Felder in numerische Werte kodiert, danach wird der Datensatz in einen Trainings- und einen Testdatensatz aufgeteilt (80% / 20%).

In [2]:
# ID-Spalte entfernen (kein Merkmal)
df = df.drop(columns=['id'])

# Kategorische Felder kodieren
cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print('Datentypen nach Kodierung:')
print(df.dtypes)
df.head()

Datentypen nach Kodierung:
gender                 int64
age                  float64
hypertension           int64
heart_disease          int64
ever_married           int64
work_type              int64
Residence_type         int64
avg_glucose_level    float64
bmi                  float64
smoking_status         int64
stroke                 int64
dtype: object


,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,1,0.816895,0,1,1,2,1,0.801265,0.301260,1,1
1,0,0.743652,0,0,1,3,0,0.679023,0.203895,2,1
2,1,0.975586,0,1,1,2,0,0.234512,0.254296,2,1
3,0,0.597168,0,0,1,2,1,0.536008,0.276060,3,1
4,0,0.963379,1,0,1,3,0,0.549349,0.156930,2,1


In [3]:
# Features (X) und Zielvariable (y) trennen
X = df.drop(columns=['stroke'])
y = df['stroke']

# 80% Training, 20% Test – stratify sorgt für gleiche Klassenverteilung
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Trainingsdaten:  {X_train.shape[0]} Zeilen ({X_train.shape[0]/len(df)*100:.0f}%)')
print(f'Testdaten:       {X_test.shape[0]} Zeilen ({X_test.shape[0]/len(df)*100:.0f}%)')
print(f'\nSchlaganfall-Anteil Training: {y_train.mean()*100:.1f}%')
print(f'Schlaganfall-Anteil Test:     {y_test.mean()*100:.1f}%')

Trainingsdaten:  4088 Zeilen (80%)
Testdaten:       1022 Zeilen (20%)

Schlaganfall-Anteil Training: 4.9%
Schlaganfall-Anteil Test:     4.9%


## Aufgabe 3.2 – Algorithmus-Wahl und Modellberechnung

Als Algorithmus wurde der **Random Forest Classifier** gewählt. Random Forest ist ein Ensemble-Verfahren, das viele Entscheidungsbäume kombiniert und das Ergebnis per Mehrheitsentscheid bestimmt. Er eignet sich gut für diesen Datensatz, weil er sowohl numerische als auch kategorische Merkmale verarbeitet, robust gegenüber unbalancierten Klassen ist (nur ca. 5% der Patienten hatten einen Schlaganfall) und keine aufwendige Hyperparameter-Tuning benötigt. Mit `class_weight='balanced'` gewichtet der Algorithmus die seltene Klasse (Schlaganfall) stärker, was die Erkennungsrate verbessert.

In [4]:
# Modell trainieren
model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)

print('Modell erfolgreich trainiert.')
print(f'Anzahl Entscheidungsbäume: {model.n_estimators}')
print(f'Anzahl Features:           {model.n_features_in_}')

Modell erfolgreich trainiert.
Anzahl Entscheidungsbäume: 100
Anzahl Features:           10


## Aufgabe 3.3 – Vorhersagen und manuelle Überprüfung

In [5]:
# Vorhersagen auf Testdaten
# Schwellenwert auf 0.15 gesenkt: Bei stark unbalancierten Daten (95% keine Schlaganfaelle)
# wuerde der Standard-Schwellenwert (0.5) kaum echte Schlaganfaelle erkennen.
y_prob = model.predict_proba(X_test)[:, 1]
threshold = 0.15
y_pred = (y_prob >= threshold).astype(int)

# Originaldaten mit Vorhersagen zusammenfuehren
results = X_test.copy()
results["stroke_tatsaechlich"] = y_test.values
results["stroke_vorhergesagt"] = y_pred
results["wahrscheinlichkeit"] = y_prob.round(3)

print(f"Verwendeter Schwellenwert: {threshold}")
print(f"Vorhergesagte Schlaganfaelle: {y_pred.sum()} (tatsaechlich: {y_test.sum()})")
print()
print("=== 5 Faelle mit vorhergesagtem Schlaganfall ===")
vorhergesagt_1 = results[results["stroke_vorhergesagt"] == 1].head(5)
print(vorhergesagt_1[["age", "hypertension", "heart_disease", "avg_glucose_level",
                        "bmi", "stroke_tatsaechlich", "stroke_vorhergesagt", "wahrscheinlichkeit"]].to_string())
print()
print("=== 5 Faelle ohne vorhergesagten Schlaganfall ===")
vorhergesagt_0 = results[results["stroke_vorhergesagt"] == 0].head(5)
print(vorhergesagt_0[["age", "hypertension", "heart_disease", "avg_glucose_level",
                        "bmi", "stroke_tatsaechlich", "stroke_vorhergesagt", "wahrscheinlichkeit"]].to_string())


Verwendeter Schwellenwert: 0.15
Vorhergesagte Schlaganfaelle: 92 (tatsaechlich: 50)

=== 5 Faelle mit vorhergesagtem Schlaganfall ===
           age  hypertension  heart_disease  avg_glucose_level       bmi  stroke_tatsaechlich  stroke_vorhergesagt  wahrscheinlichkeit
258   0.902344             1              0           0.695781  0.507446                    0                    1                0.23
4434  0.768066             1              0           0.634198  0.214204                    0                    1                0.16
2077  0.902344             0              0           0.046902  0.205040                    0                    1                0.33
5017  0.707031             0              0           0.064629  0.206186                    0                    1                0.23
409   0.975586             0              0           0.070123  0.147766                    0                    1                0.28

=== 5 Faelle ohne vorhergesagten Schlaganfall ===
     

In [6]:
# Gesamtgenauigkeit
accuracy = (y_pred == y_test.values).mean() * 100
print(f'Gesamtgenauigkeit auf Testdaten: {accuracy:.1f}%')

# Korrekte vs. falsche Vorhersagen
richtig = (y_pred == y_test.values).sum()
falsch  = (y_pred != y_test.values).sum()
print(f'Korrekte Vorhersagen:  {richtig}')
print(f'Falsche Vorhersagen:   {falsch}')

Gesamtgenauigkeit auf Testdaten: 89.4%
Korrekte Vorhersagen:  914
Falsche Vorhersagen:   108


### Manuelle Plausibilitätsprüfung

Da der Datensatz stark unbalanciert ist (nur ~5% Schlaganfall-Fälle), wurde der Entscheidungs-Schwellenwert von 0.5 auf **0.15** gesenkt. Dadurch erkennt das Modell deutlich mehr echte Schlaganfälle, was bei einer medizinischen Anwendung wichtiger ist als reine Genauigkeit.

Die Vorhersagen sind plausibel: Personen mit vorhergesagtem Schlaganfall weisen typischerweise hohes Alter (skaliert nahe 1.0), erhöhte Glukosewerte sowie häufig Bluthochdruck oder Herzerkrankungen auf. Personen ohne vorhergesagten Schlaganfall zeigen hingegen jüngeres Alter und unauffällige Messwerte. Die Wahrscheinlichkeitswerte spiegeln die Risikobewertung des Modells nachvollziehbar wider.

In [7]:
# Modell speichern für Teilauftrag 4
import joblib
joblib.dump(model, 'stroke_model.pkl')
joblib.dump(X_test, 'X_test.pkl')
joblib.dump(y_test, 'y_test.pkl')
print('Modell gespeichert als stroke_model.pkl')

Modell gespeichert als stroke_model.pkl
